In [2]:
import numpy as np
import h5py
import cv2
from joblib import load

import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from mpl_toolkits.mplot3d import Axes3D
import mpl_toolkits.mplot3d.axes3d as p3
import matplotlib.animation as animation

%matplotlib widget
import time
%load_ext autoreload
import os
%autoreload 2

from ipywidgets import Video, Image, VBox, Text
from sidecar import Sidecar
from IPython.display import display
from scipy.interpolate import interp1d
from scipy import interpolate
import numpy as np
import os
import cv2
import base64
import warnings

from ipywidgets import interact, interactive, fixed, interact_manual, FloatSlider, Layout, AppLayout
import ipywidgets as widgets
import sklearn

# Our custom-made imports
import sys
sys.path.append('./lib/')
from DataManagement import SplitdataManager, Filepaths, Calibration

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
### ----- Introduction ---- ####
#
# In this notebook, we will make a movie with two panels.
#
# This movie will be based on a subset of the frames from a single splitdata,
# although it is extendable to more.
#
# The left panel will be a splitdata movie from the XY view, with image coordinates of the skeletons drawn on top.
# This could easily be replaced with the idTracker coordinates if you like
#
# The right panel will be a 3D animation of the fish swimming
#
# The approach will be to make these two movies separately, and then combine them.
# This final combination step is quick.
#
# I have presented this information in a notebook, as opposed to a python text file,
# to hopefully make it more adaptable. All the details are available here to hopefully allow you to adapt
# this to whatever purpose you like. 
# 
# Ask Liam questions on slack

In [3]:
# --- start by loading the tracking information --- #

# We will  load the 3D coordinates, after processing (but raw is also OK),
# and the image coordinates


# load the tracking results
trackPath = '/home/liam/fish_analysis/nugget_dump/20201201/FishTank20200130_153857_tracks_A.h5'
with h5py.File(trackPath, 'r') as hf:
    #tracks_3D_raw = hf['tracks_3D_raw'][:]
    tracks_imCoords_raw = hf['tracks_imCoords_raw'][:]
    tracks_3D_smooth = hf['tracks_3D_smooth'][:]
    
print(tracks_3D_smooth.shape)
print(tracks_imCoords_raw.shape)
print()

numFrames, numFish, numBodyPoints, _ = tracks_3D_smooth.shape
print(numFrames)
print(numFish)
print(numBodyPoints)

(501943, 2, 3, 3)
(3, 501943, 2, 3, 2)

501943
2
3


In [4]:
# ---- Make a splitdata manager --- #

# we will use this object to easily find the paths of the splitdata.mp4s, that we will want to read-in

# this folder is the same as we use in the 3D GUI -  a folder containing the D_XZ (and other cam) folders, which in turn contain the splitdata movies
experiment_path = '/media/stephens-group/guest_drive/labelling/complete_shortfin_experiment/FishTank20200130_153857/'
splitman = SplitdataManager(experiment_path)

In [5]:
# derive some extra information
# feel free to explore what these contain
# These are various ways of moving between splitdata_index, splitdata_name, and getting filepaths etc

splitdata_names = [triplet[0].split('/')[-1] for triplet in splitman.splitdata_paths]
splitdata_idxs = [i for i in range(len(splitdata_names))]
splitdata_name_to_idx_dict = dict(zip(splitdata_names, splitdata_idxs))
splitdata_name_to_paths = dict(zip(splitdata_names, splitman.splitdata_paths))

In [6]:
# set the index of the splitdata file we are interested in
splitdata_idx = 0

In [8]:
# #  Alternatively.

# # find the index of a particular splitdata name
# splitdata_name = 'splitdata0009'
# print(splitdata_name_to_idx_dict[splitdata_name])
# print()

# # find the movie paths from a splitdata name
# print(splitdata_name_to_paths[splitdata_name])

In [7]:
# get the tracking results for this splitdata
splitdata_frame_range = splitman.start_stop_frames_for_splitdata[splitdata_idx]
splitdata_frame_range

array([   0, 6000])

In [8]:
# parse the tracking results down to the section of data corresponding to this splitdata movie
splitdata_tracking_results = np.copy(tracks_3D_smooth[splitdata_frame_range[0]:splitdata_frame_range[1]])
splitdata_raw_imageCoords = np.copy(tracks_imCoords_raw[:, splitdata_frame_range[0]:splitdata_frame_range[1]])
print(splitdata_tracking_results.shape)
print(splitdata_raw_imageCoords.shape)

(6000, 2, 3, 3)
(3, 6000, 2, 3, 2)


In [9]:
# in this case, we want the XY splitdata.mp4 path.

# use the splitdata_idx and the SplitdataManager object to get the filepath
splitdata_movie_path = splitman.splitdata_paths[splitdata_idx][1] + '.mp4'
splitdata_movie_path

'/media/stephens-group/guest_drive/labelling/complete_shortfin_experiment/FishTank20200130_153857/E_xy/splitdata0009.mp4'

In [12]:
# # alternatively, you could use name of the splitdata that you want
# splitdata_name = 'splitdata0009'
# splitdata_idx = splitdata_name_to_idx_dict[splitdata_name]
# splitdata_movie_path = splitdata_name_to_paths[splitdata_name][1]+'.mp4'
# splitdata_movie_path

In [10]:

# ---- set the path to a folder which will be used to store the output ----#

# Ultimately there will be three movies written to this folder


saveFolder = '/home/liam/temp/movie_folder'


# ------------------------------------#

if not os.path.exists(saveFolder):
    os.makedirs(saveFolder)
    
    
# from these inputs derive the other information that we need
splitdata_mov_with_bodypoints_path = os.path.join(saveFolder, 'splitdata_mov.mp4')
animation_path = os.path.join(saveFolder, '3D_animation.mp4')
final_movie_path = os.path.join(saveFolder, 'movie.mp4')

In [14]:
# --- panel 1: the splitdata movie with bodypoints drawn ----- #

In [11]:
def draw_skeleton_on_camera_frame(frame, frame_image_coordinates):
    ''' Draw the image coordinates for the fish on the frame.
    
    --- Inputs ---
    frame: array, an image frame taken from a .mp4
    frame_image_coordinates: shape (numFish, numBodyPoints, 2)
    
    --- Returns ---
    frame: in input frame with the skeleton drawn
    '''
    # parse shapes from the input
    numFish, numBodyPoints, _ = frame_image_coordinates.shape
    
    # how bodypoints are joined  
    edges = [[0,1], [1,2]]   
    
    # colors of the fish
    fish_colors = [(255,0,0), (0,0,255)]
    
    for fishIdx in range(numFish):

        # -- draw the bodypoints -- #
        for bpIdx in range(numBodyPoints):
            # get the data
            imCoord = frame_image_coordinates[fishIdx, bpIdx]
            # get the color
            fcolor = fish_colors[fishIdx]
            color = ( int (fcolor [ 0 ]), int(fcolor [ 1 ]), int(fcolor [ 2 ]) )
            # if we don't have a NaN, draw the point
            if np.all(np.isnan(imCoord)):
                continue
            cv2.circle(frame, (int(imCoord[0]), int(imCoord[1])), radius=2, color=color, thickness=-1)

        # --- draw the lines between bodypoints -- #
        for edge in edges:
            pt1_bpIdx = edge[0]
            pt2_bpIdx = edge[1]

            pt1_imCoord_x = frame_image_coordinates[fishIdx][pt1_bpIdx][0]
            pt1_imCoord_y = frame_image_coordinates[fishIdx][pt1_bpIdx][1]
            pt1_imCoord = np.array([pt1_imCoord_x, pt1_imCoord_y])

            pt2_imCoord_x = frame_image_coordinates[fishIdx][pt2_bpIdx][0]
            pt2_imCoord_y = frame_image_coordinates[fishIdx][pt2_bpIdx][1]
            pt2_imCoord = np.array([pt2_imCoord_x, pt2_imCoord_y])

            # parse the points for format and emptiness
            if np.all(np.isnan(pt1_imCoord)):
                pt1 =tuple([np.NaN, np.NaN])
            else:
                pt1 = tuple([int(x) for x in pt1_imCoord])
            if np.all(np.isnan(pt2_imCoord)):
                pt2 =tuple([np.NaN, np.NaN])
            else:
                pt2 = tuple([int(x) for x in pt2_imCoord])
                
            # if all is good, draw the lines
            if ~np.all(np.isnan(pt1)) and ~np.all(np.isnan(pt2)):
                cv2.line(frame, pt1=pt1,  pt2=pt2, color=color, thickness=1, lineType=cv2.LINE_AA)
                
    return frame

In [12]:


t0 = time.time()

# pick a start and stop frame (within the frames for this splitdata folder - not the entire experiment)
startFrame = 0
stopFrame = 1000

# set other parameters for the movie
fps = 100
fourcc = cv2.VideoWriter_fourcc('m','p','4','v')
# THIS NEED TO MATCH THE SHAPE OF THE FRAMES YOU MAKE
nrows = 1024
ncols = 1280
resolution = (ncols,nrows) 
out = cv2.VideoWriter(splitdata_mov_with_bodypoints_path, fourcc, fps, resolution)



# open up the video we want to read frames from
cap = cv2.VideoCapture(splitdata_movie_path)


# loop over the frames that we want, 
# grabbing each frame, drawing the bodypoints on it, and then saving the edited frame to the new movie
for fIdx in range(startFrame, stopFrame):
    
    # print an update
    if np.mod(fIdx, 10) == 0:
        print(fIdx)
        
    # move cap to current frame
    cap.set(1,fIdx)
    
    # read the frame
    ret, frame = cap.read()
    if not ret:
        raise TypeError('problem with splitdata_movie_path in reading the frames')
        
    # convert to color
    #movie_panel = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
    # Draw the skeletons
    camIdx = 1 # for XY
    frame_image_coordinates = np.copy(splitdata_raw_imageCoords[camIdx, fIdx])
    frame = draw_skeleton_on_camera_frame(frame, frame_image_coordinates)
    
    # convert to BGR before writing the frame, otherwise the colors will be swapped
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    
    # write the frame
    out.write(frame)
    
    
# close the writer and reader
cap.release()
out.release()


print()
print(time.time()-t0)

0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290
300
310
320
330
340
350
360
370
380
390
400
410
420
430
440
450
460
470
480
490
500
510
520
530
540
550
560
570
580
590
600
610
620
630
640
650
660
670
680
690
700
710
720
730
740
750
760
770
780
790
800
810
820
830
840
850
860
870
880
890
900
910
920
930
940
950
960
970
980
990

71.83109045028687


In [14]:
# -------------- make the 3D animation --------------------- #



def animate_scatters(iteration, data, scatters, lines, scatter_projections, lines_projections, text, all_timestamps):
    """
    Update the data held by the scatter plot and therefore animates it.

    Args:
        iteration (int): Current iteration of the animation
        data (list): List of the data positions at each iteration.
        scatters (list): List of all the scatters (One per element)

    Returns:
        list: List of scatters (One per element) with new coordinates
    """
    #if np.mod(iteration, 100) == 0:
        #print(iteration)
    # parse the input shapes
    numFrames, numFish, numBodyPoints, _ = data.shape
 
    # update the scatters
    for fishIdx in range(numFish):
        for bpIdx in range(numBodyPoints):
            scatters[fishIdx][bpIdx]._offsets3d = (data[iteration,fishIdx,bpIdx,0:1],
                                                   data[iteration,fishIdx,bpIdx,1:2],
                                                   data[iteration,fishIdx,bpIdx,2:])
            
    # update the lines
    for fishIdx,line in enumerate(lines):
        line.set_data(data[iteration, fishIdx, :, :2].swapaxes(0,1))
        line.set_3d_properties(data[iteration, fishIdx, :, 2])
        
        
    # update the text
    text.set_text(all_timestamps[iteration])
        
    # -------------- #
    for projectionIdx in range(3):
        
        # XZ => Y=0
        if projectionIdx == 0:
            projectionData = np.copy(data[iteration])
            projectionData[:, :, 1] = 0
        # XY => Z=0
        elif projectionIdx == 1:
            projectionData = np.copy(data[iteration])
            projectionData[:, :, 2] = 0
        # YZ => X=0
        elif projectionIdx == 2:
            projectionData = np.copy(data[iteration])
            projectionData[:, :, 0] = 0
            
        proj_scatters = scatter_projections[projectionIdx]
        proj_lines = lines_projections[projectionIdx]
        
        # update the scatters
        for fishIdx in range(numFish):
            for bpIdx in range(numBodyPoints):
                proj_scatters[fishIdx][bpIdx]._offsets3d = (projectionData[fishIdx,bpIdx,0:1],
                                                            projectionData[fishIdx,bpIdx,1:2],
                                                            projectionData[fishIdx,bpIdx,2:])
        # update the lines
        for fishIdx,line in enumerate(proj_lines):
            line.set_data(projectionData[fishIdx, :, :2].swapaxes(0,1))
            line.set_3d_properties(projectionData[fishIdx, :, 2])

    return




def make_movie(data, f0, fE, fps, savePath='/home/liam/temp/testanimation.mp4'):
    """
    Creates the 3D figure and animates it with the input data.

    Args:
        data (list): List of the data positions at each iteration.
        save (bool): Whether to save the recording of the animation. (Default to False).
    """
    data = data[f0:fE]
    
    # parse the input shapes
    numFrames, numFish, numBodyPoints, _ = data.shape
    fish_colors = ['red', 'blue']

    # Attaching 3D axis to the figure
    fig = plt.figure()
    fig.tight_layout()
    ax = p3.Axes3D(fig, [0.,0.,1.,1.])

    # Initialize scatters (list over fish, list over bps)
    symbols = ['o', 's', 'x']
    sizes=12
    
    # Main scatters
    scatters = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        fish_scatters = []
        for bpIdx in range(numBodyPoints):
            fish_scatters.append(ax.scatter(data[0,fishIdx,bpIdx,0:1], data[0,fishIdx,bpIdx,1:2], data[0,fishIdx,bpIdx,2:], 
                                            c=col, s=sizes, marker=symbols[bpIdx])) 
        scatters.append(fish_scatters)
        
    # Main lines
    lines = []
    for fishIdx in range(numFish):
        col = fish_colors[fishIdx]
        line = ax.plot(data[0, fishIdx, :, 0], data[0, fishIdx, :, 1], data[0, fishIdx, :, 2], c=col)[0]
        lines.append(line)
        
    # ----------------
    # projections
    symbols = ['x', 'o', 's']
    sizes=5

    scatter_projections = []
    lines_projections = []
    for projectionIdx in range(3):
        proj_scatters = []
        proj_lines = []
        for fishIdx in range(numFish):
            col = fish_colors[fishIdx]
            # scatter plots
            fish_scatters = []
            for bpIdx in range(numBodyPoints):
                #print(symbols[bpIdx])
                fish_scatters.append(ax.scatter(data[0,fishIdx,bpIdx,0:1], data[0,fishIdx,bpIdx,1:2], data[0,fishIdx,bpIdx,2:], 
                                                c=col, s=sizes, marker=symbols[bpIdx], alpha=0.1))
            # lines
            line = ax.plot(data[0, fishIdx, :, 0], data[0, fishIdx, :, 1], data[0, fishIdx, :, 2], c=col, alpha=0.1)[0]
            proj_lines.append(line)
            proj_scatters.append(fish_scatters)
        
        scatter_projections.append(proj_scatters)
        lines_projections.append(proj_lines)
    # ----------------
    
    # the text
    all_timestamps = ['frame: '+ str(i).zfill(7) for i in range(f0, fE)]
    time_text = ax.text(14, 0, 43,'', fontsize=15)

    # Setting the axes properties
    ax.set_xlim3d([0, 40])
    ax.set_xlabel('X')

    ax.set_ylim3d([0, 40])
    ax.set_ylabel('Y')

    ax.set_zlim3d([0, 40])
    ax.set_zlabel('Z')

    # Provide starting angle for the view
    elevation_angle = 10
    azimuthal_angle = 45
    ax.view_init(elevation_angle, azimuthal_angle)

    ani = animation.FuncAnimation(fig, animate_scatters, numFrames, fargs=(data, scatters, lines, 
                                                                           scatter_projections, lines_projections,
                                                                           time_text, all_timestamps),
                                       interval=10, blit=False, repeat=True)

    Writer = animation.writers['ffmpeg']
    writer = Writer(fps=fps, metadata=dict(artist='LOS'), bitrate=3500)
    ani.save(savePath, writer=writer)
    return fig



# make the movie

t0 = time.time()

#f0 = splitdata_frame_range[0] + startFrame
#fE = splitdata_frame_range[0] + startFrame + stopFrame

f0 = 0
fE = 1

fig = make_movie(tracks_3D_smooth, f0, fE, fps, savePath=animation_path)

tE = time.time()
print(tE-t0)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

0.46073436737060547


In [18]:
# ----------- now we want to combine the two movies into 1, by reading the frames from both and sticking them together ---------------- #


# the first thing to do is to read a frame from each movie, stick them together and see how big they are,
# so we know how to set the resolution of the final output movie

panel1_cap = cv2.VideoCapture(splitdata_mov_with_bodypoints_path)
panel2_cap = cv2.VideoCapture(animation_path)

panel1_ret, panel1_frame = panel1_cap.read()
panel2_ret, panel2_frame = panel2_cap.read()

panel1_cap.release()
panel2_cap.release()


In [19]:
panel1_frame.shape

(1024, 1280, 3)

In [20]:
panel2_frame.shape

(480, 640, 3)

In [21]:
# these images are different sizes. So we have a choice as to how to combine them.
# here i will chose to downsample panel1 to match panel2, but you could change the size of one or both
# it is important here to figure out how you would like to combine the images

panel1_frame_resized = cv2.resize(panel1_frame, (640,480), interpolation=cv2.INTER_AREA) 
panel1_frame_resized.shape

(480, 640, 3)

In [22]:
# view the resized frame if you want
plt.figure()
plt.imshow(panel1_frame_resized)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [23]:
# now check the size of the combined frames

combined_frame_sample = np.hstack([panel1_frame_resized, panel2_frame])
plt.figure()
plt.imshow(combined_frame_sample)
print()
print(combined_frame_sample.shape)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …


(480, 1280, 3)


In [24]:
# now combine the 2 movies
t0 = time.time()

# make the combined movie
panel1_cap = cv2.VideoCapture(splitdata_mov_with_bodypoints_path)
panel2_cap = cv2.VideoCapture(animation_path)

fourcc = cv2.VideoWriter_fourcc('m','p','4','v')
# THIS NEED TO MATCH THE SHAPE OF THE FRAMES YOU MAKE
resolution = (1280, 480) 

out = cv2.VideoWriter(final_movie_path, fourcc, fps, resolution)

nfs = stopFrame - startFrame
for ii in range(nfs):
    
    if panel1_cap.isOpened() and panel2_cap.isOpened():  
        
        ret_p1, frame_p1 = panel1_cap.read()
        if not ret_p1:
            raise TypeError('Problem with panel1 movie')
            
        ret_p2, frame_p2 = panel2_cap.read()
        if not ret_p2:
            raise TypeError('Problem with panel2 movie')

        # combine the frames
        panel1_frame_resized = cv2.resize(frame_p1, (640,480), interpolation=cv2.INTER_AREA) 
        combined_frame = np.hstack([panel1_frame_resized, frame_p2])
        
        # convert color
        # This time we don't need to convert color, since we are loading and saving
        # in the same format
        
        # write the frame
        out.write(combined_frame)
    
    
panel1_cap.release()
panel2_cap.release()
out.release()

tE = time.time()
print('Fin: {0}'.format(tE-t0))



Fin: 9.747287511825562


In [25]:
# now you can view the final movie at
final_movie_path

'/home/liam/temp/movie_folder/movie.mp4'